# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row (in the raw warehouse table) = one report_date × client_hash_id × content_hash_id daily fact one content item's search performance on one day for one client. My lane rolls this up to one row = one (client_hash_id, content_hash_id) pair's March 2026 summary, built by aggregating all its daily rows where month = '2026-03'. March 2026 is a mid-panel month not the sealed final month (fact_content_daily_performance_sample = June 2026).

In [26]:
%pip -q install duckdb
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_k.....): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
MONTH = '2026-03'

con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) mn, MAX(report_date) mx FROM {FACT} WHERE month = '{MONTH}'").df()

Paste your Hugging Face READ token (hf_k.....): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,mn,mx
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I touch this notebook, sorted into one bucket

In [18]:
import pandas as pd

field_classification = {
    "content_hash_id": "context — join/grouping only, never a feature",
    "client_hash_id": "context — join/grouping only; used for per-client checks and grouped splits",
    "report_date": "context — defines the window, not a feature itself",
    "gsc_impressions": "feature — observed daily search impressions, known as it happens",
    "gsc_clicks": "feature — observed daily search clicks",
    "gsc_avg_position": "feature — observed daily average rank; 0/NULL = no position data that day",
    "ga4_data_available": "context — flag to filter rows before a client's GA4 start; not a model input",
    "is_declining (built by me)": "label/proxy — computed FROM impressions across two sub-windows of March; never a feature",
    "product decision flags (health_score, priority_score, action_type)": "excluded — not shipped in this release; would be circular if it were",
}
pd.Series(field_classification).to_frame("bucket")

,bucket
content_hash_id,"context — join/grouping only, never a feature"
client_hash_id,context — join/grouping only; used for per-cli...
report_date,"context — defines the window, not a feature it..."
gsc_impressions,"feature — observed daily search impressions, k..."
gsc_clicks,feature — observed daily search clicks
gsc_avg_position,feature — observed daily average rank; 0/NULL ...
ga4_data_available,context — flag to filter rows before a client'...
is_declining (built by me),label/proxy — computed FROM impressions across...
"product decision flags (health_score, priority_score, action_type)",excluded — not shipped in this release; would ...


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries: grain, slice size + date span, availability.

In [19]:
###Query 1,grain:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) c
    FROM {FACT}
    WHERE month = '{MONTH}'
    GROUP BY 1,2
    HAVING c > 31
    LIMIT 5
""").df()
print("Rows violating grain (should be empty):", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating grain (should be empty): 0


,client_hash_id,content_hash_id,c


In [20]:
###Query 2, slice size + date span:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS daily_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {FACT}
    WHERE month = '{MONTH}'
""").df()
slice_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,daily_rows,n_clients,n_content,min_d,max_d
0,9841378,55,331437,2026-03-01,2026-03-31


In [21]:
###Query 3, availability (IS TRUE):
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {FACT}
    WHERE month = '{MONTH}'
""").df()
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


Five features, built from the same month, each with an "available when?" line.

In [22]:
feat = con.sql(f"""
    WITH bounds AS (SELECT DATE '{MONTH}-16' AS mid),
    agg AS (
        SELECT f.client_hash_id, f.content_hash_id,
            SUM(f.gsc_impressions) AS impressions_month,
            SUM(f.gsc_clicks) AS clicks_month,
            AVG(NULLIF(f.gsc_avg_position, 0)) AS avg_position_month,
            COUNT(*) FILTER (WHERE f.gsc_impressions > 0) AS active_days_month,
            SUM(CASE WHEN f.report_date < b.mid THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN f.report_date >= b.mid THEN f.gsc_impressions ELSE 0 END) AS imp_second_half
        FROM {FACT} f, bounds b
        WHERE f.month = '{MONTH}'
        GROUP BY 1,2
        HAVING SUM(f.gsc_impressions) > 0
    )
    SELECT *, ROUND(clicks_month::DOUBLE / NULLIF(impressions_month,0), 4) AS ctr_month
    FROM agg
""").df()
print(f"{len(feat):,} (client, content) rows with impressions in {MONTH}")
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 (client, content) rows with impressions in 2026-03


,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position_month,active_days_month,imp_first_half,imp_second_half,ctr_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,31,4173.0,2350.0,0.0011
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255,31,245.0,208.0,0.0000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,31,3705.0,1925.0,0.0011
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,31,2440.0,2504.0,0.0026
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,23.314103,21,14.0,28.0,0.0000


**THE FIVE FEATURES**


1.   impressions_month: knowable because it's the sum of March's own observed daily impressions; nothing from April or later touches it.
2.   clicks_month: knowable because it's purely observed March search clicks.
3. avg_position_month: knowable because each day's average rank is known that day; averaged only over March.
4. active_days_month: knowable because it just counts March days with any impression, available the moment March ends.
5. ctr_month: knowable because it's a ratio of the two March-only sums above; still resolved at March's close.




The trap: add one label-derived column on purpose, watch the score jump, then delete it.

In [23]:
###the leakage trap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_df = feat.dropna(subset=['avg_position_month']).copy()
model_df['is_declining'] = (model_df['imp_second_half'] < 0.8 * model_df['imp_first_half']).astype(int)

honest_features = ['impressions_month', 'clicks_month', 'avg_position_month', 'active_days_month', 'ctr_month']

def quick_score(df, cols):
    X, y = df[cols], df['is_declining']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, m.predict_proba(Xte)[:,1])

print("Honest AUC (5 features only):", round(quick_score(model_df, honest_features), 3))

# THE TRAP — a column derived directly from the label's own formula
model_df['imp_second_half_LEAK'] = model_df['imp_second_half']
leaky_features = honest_features + ['imp_second_half_LEAK']
print("Leaky AUC (with label-derived column):", round(quick_score(model_df, leaky_features), 3))

# delete it — keep only the honest number
del model_df['imp_second_half_LEAK']
print("Final honest AUC (kept):", round(quick_score(model_df, honest_features), 3))

Honest AUC (5 features only): 0.58
Leaky AUC (with label-derived column): 1.0
Final honest AUC (kept): 0.58


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell you:

In [24]:
print(
    f"Named limitation: only {avail['pct_available'][0]}% of March rows have "
    "ga4_data_available = TRUE, so any GA4-based feature (sessions, engagement) would "
    "silently drop most of the panel or misrepresent clients who onboarded to GA4 late — "
    "this slice is GSC-complete but GA4-sparse. It also can't tell us WHY a page declined, "
    "only THAT it declined within one month's own two halves — that's a proxy, not a "
    "forward-looking outcome label."
)

Named limitation: only 4.2% of March rows have ga4_data_available = TRUE, so any GA4-based feature (sessions, engagement) would silently drop most of the panel or misrepresent clients who onboarded to GA4 late — this slice is GSC-complete but GA4-sparse. It also can't tell us WHY a page declined, only THAT it declined within one month's own two halves — that's a proxy, not a forward-looking outcome label.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.